### ***Exploratory Data Analysis***

In [2]:
# lets download data\
%load_ext sql
import csv
import sqlite3
import prettytable
import pandas as pd
prettytable.DEFAULT='default'

In [3]:
# load data set 
dataframe=pd.read_csv(r'D:\Data Science Project\Exploratory Data Analysis\Spacex.csv')

In [4]:
#lets go for database
conn=sqlite3.connect('database.db')
cursor=conn.cursor()
%sql sqlite:///database.db

In [5]:
# dataframe to sql
dataframe.to_sql('SPACEXTBL',con=conn,if_exists='replace',index=False,method='multi')

101

In [6]:
# Lets create a table
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

 * sqlite:///database.db
(sqlite3.OperationalError) table SPACEXTABLE already exists
[SQL: create table SPACEXTABLE as select * from SPACEXTBL where Date is not null]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [7]:
# select unique launch sites in the space mission
%sql select distinct(Launch_Site) from spacextable 

 * sqlite:///database.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


In [8]:
%sql select Landing_Outcome from spacextable

 * sqlite:///database.db
Done.


Landing_Outcome
Failure (parachute)
Failure (parachute)
No attempt
No attempt
No attempt
Uncontrolled (ocean)
No attempt
No attempt
Controlled (ocean)
Controlled (ocean)


In [9]:
# five records where launch sites begin with 'CCA'
%sql select * from spacextable where Launch_Site Like 'CCA%' LIMIT 5

 * sqlite:///database.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


In [10]:
# total mass carried by booster launched by NASA (CRS)
%sql select SUM(PAYLOAD_MASS__KG_) as PayloadbyNASA from spacextable where Customer='NASA (CRS)' 

 * sqlite:///database.db
Done.


PayloadbyNASA
45596


In [11]:
# average payload mass carried by booster version F9 V1.1
%sql select avg(PAYLOAD_MASS__KG_) as PayloadByF9 from spacextable where Booster_version='F9 v1.1'

 * sqlite:///database.db
Done.


PayloadByF9
2928.4


In [12]:
# first date landing outcome ground pad
%sql select Date from spacextable where Landing_Outcome='Success (ground pad)' order by Date ASC Limit 1

 * sqlite:///database.db
Done.


Date
2015-12-22


In [13]:
# names of booster with given conditions
%sql select distinct(Booster_Version) from spacextable where Landing_Outcome='Success (drone ship)' and 4000<PAYLOAD_MASS__KG_<6000

 * sqlite:///database.db
Done.


Booster_Version
F9 FT B1021.1
F9 FT B1022
F9 FT B1023.1
F9 FT B1026
F9 FT B1029.1
F9 FT B1021.2
F9 FT B1029.2
F9 FT B1036.1
F9 FT B1038.1
F9 B4 B1041.1


In [14]:
# TOTAL SUCCESS AND FAILURE
%sql select mission_outcome,COUNT(*) as total from spacextable group by mission_outcome

 * sqlite:///database.db
Done.


Mission_Outcome,total
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


In [15]:
# total sucess and failure
%sql SELECT Booster_Version from spacextable where PAYLOAD_MASS__KG_= (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)

 * sqlite:///database.db
Done.


Booster_Version
F9 B5 B1048.4
F9 B5 B1049.4
F9 B5 B1051.3
F9 B5 B1056.4
F9 B5 B1048.5
F9 B5 B1051.4
F9 B5 B1049.5
F9 B5 B1060.2
F9 B5 B1058.3
F9 B5 B1051.6


In [16]:
dataframe

,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt
...,...,...,...,...,...,...,...,...,...,...
96,2020-11-05,23:24:23,F9 B5B1062.1,CCAFS SLC-40,"GPS III-04 , Crew-1",4311,MEO,USSF,Success,Success
97,2020-11-16,0:27:00,F9 B5B1061.1,KSC LC-39A,"Crew-1, Sentinel-6 Michael Freilich",12500,LEO (ISS),NASA (CCP),Success,Success
98,2020-11-21,17:17:08,F9 B5B1063.1,VAFB SLC-4E,"Sentinel-6 Michael Freilich, Starlink 15 v1.0",1192,LEO,NASA / NOAA / ESA / EUMETSAT,Success,Success
99,2020-11-25,2:13:00,F9 B5 B1049.7,CCAFS SLC-40,"Starlink 15 v1.0, SpaceX CRS-21",15600,LEO,SpaceX,Success,Success


In [23]:
%sql select substr(Date,6,2) as month,Landing_Outcome,Booster_Version,Launch_Site from spacextable where substr(Date,0,5)='2015' and Landing_Outcome='Failure (drone ship)'

 * sqlite:///database.db
Done.


month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


In [26]:
# rank the count of landing outcomes between specific dates
%sql select Landing_Outcome, count(*) as total from spacextable where Date between '2010-06-04' and '2017-03-20' group by Landing_Outcome order by total desc

 * sqlite:///database.db
Done.


Landing_Outcome,total
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
